In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ, scaler=None, imputer=None):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    # Aggregate features
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    # Original pairwise features
    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
    main_features.extend([f'ME_prod_M*_E*', f'ME_ratio_M*_E*'])
            
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    main_features.extend([f'MI_prod_M*_I*', f'MI_ratio_M*_I*', f'MI_spread_M*_I*'])
        
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    main_features.extend([f'MP_prod_M*_P*', f'MP_ratio_M*_P*'])
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    main_features.extend([f'MV_ratio_M*_V*', f'MV_prod_M*_V*'])
        
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.extend([f'MS_prod_M*_S*', f'MS_weighted_M*_S*'])
        
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.extend([f'EI_diff_E*_I*', f'EI_ratio_E*_I*', f'EI_prod_E*_I*'])
        
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.extend([f'EP_prod_E*_P*', f'EP_ratio_E*_P*'])
        
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.extend([f'EV_prod_E*_V*', f'EV_uncertainty_E*_V*'])
        
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.extend([f'ES_prod_E*_S*', f'ES_diff_E*_S*'])
        
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.extend([f'IP_prod_I*_P*', f'IP_discount_I*_P*'])
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.extend([f'PV_prod_P*_V*', f'PV_stability_P*_V*'])
        
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.extend([f'PS_prod_P*_S*', f'PS_contrarian_P*_S*'])
        
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.extend([f'VS_prod_V*_S*', f'VS_panic_V*_S*'])

    # Original multi-feature combinations
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 
    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 
    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 
    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']
    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    # ========== NEW ADVANCED FEATURES ==========
    
    # Market Dynamics Advanced Features
    data['MD_momentum_M*_D*'] = data['M*'] * data['D*']
    data['MD_signal_M*_D*'] = data['M*'] / (data['D*'] + 1e-8)
    main_features.extend(['MD_momentum_M*_D*', 'MD_signal_M*_D*'])
    
    # Economic-Interest Rate Interaction (real rates, growth vs rates)
    data['ED_macro_signal_E*_D*'] = data['E*'] * data['D*']
    data['ID_rate_regime_I*_D*'] = data['I*'] * data['D*']
    main_features.extend(['ED_macro_signal_E*_D*', 'ID_rate_regime_I*_D*'])
    
    # Price-Dummy (valuation regime)
    data['PD_value_regime_P*_D*'] = data['P*'] * data['D*']
    data['PD_cheap_signal_P*_D*'] = data['P*'] / (data['D*'] + 1e-8)
    main_features.extend(['PD_value_regime_P*_D*', 'PD_cheap_signal_P*_D*'])
    
    # Volatility-Dummy (vol regime)
    data['VD_vol_regime_V*_D*'] = data['V*'] * data['D*']
    main_features.append('VD_vol_regime_V*_D*')
    
    # Sentiment-Dummy (sentiment regime)
    data['SD_sentiment_regime_S*_D*'] = data['S*'] * data['D*']
    main_features.append('SD_sentiment_regime_S*_D*')
    
    # Risk Premium: Economic strength vs Interest rates
    data['risk_premium_E*_I*'] = (data['E*'] - data['I*']) / (abs(data['I*']) + 1e-8)
    main_features.append('risk_premium_E*_I*')
    
    # Valuation Adjusted for Rates (P/E type ratio)
    data['val_adj_rates_P*_I*'] = data['P*'] * (1 + data['I*'])
    data['real_valuation_P*_I*'] = data['P*'] / (1 + abs(data['I*']) + 1e-8)
    main_features.extend(['val_adj_rates_P*_I*', 'real_valuation_P*_I*'])
    
    # Risk-Adjusted Market Momentum
    data['risk_adj_momentum_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data['sharpe_proxy_M*_V*'] = (data['M*'] - data['M*'].mean()) / (data['V*'] + 1e-8)
    main_features.extend(['risk_adj_momentum_M*_V*', 'sharpe_proxy_M*_V*'])
    
    # Economic-Sentiment Alignment
    data['econ_sentiment_align_E*_S*'] = data['E*'] * data['S*'] / (abs(data['E*']) + abs(data['S*']) + 1e-8)
    main_features.append('econ_sentiment_align_E*_S*')
    
    # Market-Price Divergence (momentum vs valuation)
    data['momentum_val_diverge_M*_P*'] = abs(data['M*'] - data['P*'])
    data['momentum_val_ratio_M*_P*'] = (data['M*'] + 1e-8) / (data['P*'] + 1e-8)
    main_features.extend(['momentum_val_diverge_M*_P*', 'momentum_val_ratio_M*_P*'])
    
    # Fear Index (Volatility + negative sentiment)
    data['fear_index_V*_S*'] = data['V*'] * (1 - data['S*'])
    data['panic_score_V*_S*'] = data['V*'] ** 2 * abs(data['S*'])
    main_features.extend(['fear_index_V*_S*', 'panic_score_V*_S*'])
    
    # Greed Index (Market momentum + positive sentiment - volatility)
    data['greed_index_M*_S*_V*'] = (data['M*'] * data['S*']) / (data['V*'] + 1e-8)
    main_features.append('greed_index_M*_S*_V*')
    
    # Economic Growth vs Market Performance
    data['growth_market_gap_E*_M*'] = data['E*'] - data['M*']
    data['growth_momentum_sync_E*_M*'] = data['E*'] * data['M*'] / (abs(data['E*']) + abs(data['M*']) + 1e-8)
    main_features.extend(['growth_market_gap_E*_M*', 'growth_momentum_sync_E*_M*'])
    
    # Policy Pressure (Rates vs Economic condition)
    data['policy_pressure_I*_E*'] = data['I*'] * data['E*']
    data['policy_gap_I*_E*'] = abs(data['I*']) - abs(data['E*'])
    main_features.extend(['policy_pressure_I*_E*', 'policy_gap_I*_E*'])
    
    # Valuation-Economic Justification
    data['val_justified_P*_E*'] = data['P*'] / (data['E*'] + 1e-8)
    data['earnings_yield_proxy_E*_P*'] = data['E*'] / (abs(data['P*']) + 1e-8)
    main_features.extend(['val_justified_P*_E*', 'earnings_yield_proxy_E*_P*'])
    
    # Sentiment-Price Momentum (contrarian signal)
    data['contrarian_signal_S*_M*'] = -data['S*'] * data['M*']
    data['sentiment_momentum_gap_S*_M*'] = data['S*'] - data['M*']
    main_features.extend(['contrarian_signal_S*_M*', 'sentiment_momentum_gap_S*_M*'])
    
    # Triple Interactions
    data['risk_reward_M*_P*_V*'] = (data['M*'] * data['P*']) / (data['V*'] + 1e-8)
    data['fundamental_score_E*_P*_I*'] = data['E*'] * data['P*'] / (abs(data['I*']) + 1e-8)
    data['market_condition_M*_V*_S*'] = data['M*'] * data['S*'] / (data['V*'] + 1e-8)
    data['macro_condition_E*_I*_S*'] = data['E*'] * data['S*'] / (abs(data['I*']) + 1e-8)
    main_features.extend(['risk_reward_M*_P*_V*', 'fundamental_score_E*_P*_I*', 
                          'market_condition_M*_V*_S*', 'macro_condition_E*_I*_S*'])
    
    # Quadruple Interactions
    data['total_risk_score_V*_I*_S*_D*'] = data['V*'] * abs(data['I*']) * abs(data['S*']) * data['D*']
    data['market_qual_score_M*_E*_P*_S*'] = (data['M*'] * data['E*'] * data['S*']) / (abs(data['P*']) + 1e-8)
    main_features.extend(['total_risk_score_V*_I*_S*_D*', 'market_qual_score_M*_E*_P*_S*'])
    
    # Normalized Ratios
    data['norm_market_strength'] = data['M*'] / (data['M*_E*_I*_P*_V*_S*_D*'] + 1e-8)
    data['norm_econ_strength'] = data['E*'] / (data['M*_E*_I*_P*_V*_S*_D*'] + 1e-8)
    data['norm_vol_impact'] = data['V*'] / (data['M*_E*_I*_P*_V*_S*_D*'] + 1e-8)
    data['norm_sentiment_impact'] = data['S*'] / (data['M*_E*_I*_P*_V*_S*_D*'] + 1e-8)
    main_features.extend(['norm_market_strength', 'norm_econ_strength', 
                          'norm_vol_impact', 'norm_sentiment_impact'])
    
    # Volatility-adjusted returns proxy
    data['vol_adj_market_M*_V*_squared'] = data['M*'] / (data['V*']**2 + 1e-8)
    main_features.append('vol_adj_market_M*_V*_squared')
    
    # Interest Rate Sensitivity
    data['duration_proxy_P*_I*_squared'] = data['P*'] * (data['I*']**2)
    main_features.append('duration_proxy_P*_I*_squared')
    
    # Market Regime Indicator
    data['bull_bear_indicator'] = (data['M*'] * data['S*']) - (data['V*'] * abs(data['I*']))
    main_features.append('bull_bear_indicator')
    
    # Absolute value features for regime detection
    data['abs_M*'] = abs(data['M*'])
    data['abs_E*'] = abs(data['E*'])
    data['abs_S*'] = abs(data['S*'])
    main_features.extend(['abs_M*', 'abs_E*', 'abs_S*'])
    
    # Squared features for non-linear relationships
    data['M*_squared'] = data['M*'] ** 2
    data['V*_squared'] = data['V*'] ** 2
    data['S*_squared'] = data['S*'] ** 2
    main_features.extend(['M*_squared', 'V*_squared', 'S*_squared'])
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        data[feature_cols] = imputer.transform(data[feature_cols])
    
    if typ == "train":
        scaler = MinMaxScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
        data[feature_cols] = scaler.transform(data[feature_cols])
    
    if typ == "train":
        return data, scaler, imputer
    else:
        return data


train_processed, scaler, imputer = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
exist = ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18',
         'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'S1',
       'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12',
       'I2', 'P8', 'P9', 'P10', 'P12', 'P13', 'M*', 'E*', 'I*', 'P*', 'V*',
       'S*', 'D*', 'M*_P*_V*', 'M*_E*_S*', 'M*_P*_I*_V*', 'V*_M*_E*_I*',
       'V*_S*_D*', 'E*_I*_V*_D*', 'M*_V*_S*_D*', 'P*_V*_S*', 'P*_M*_D*',
       'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*', 'ME_prod_M*_E*',
       'ME_ratio_M*_E*', 'MI_prod_M*_I*', 'MI_ratio_M*_I*', 'MI_spread_M*_I*',
       'MP_prod_M*_P*', 'MP_ratio_M*_P*', 'MV_ratio_M*_V*', 'MV_prod_M*_V*',
       'MS_prod_M*_S*', 'MS_weighted_M*_S*', 'EI_diff_E*_I*', 'EI_ratio_E*_I*',
       'EI_prod_E*_I*', 'EP_prod_E*_P*', 'EP_ratio_E*_P*', 'EV_prod_E*_V*',
       'EV_uncertainty_E*_V*', 'ES_prod_E*_S*', 'ES_diff_E*_S*',
       'IP_prod_I*_P*', 'IP_discount_I*_P*', 'IV_prod_I*_V*', 'IS_prod_I*_S*',
       'PV_prod_P*_V*', 'PV_stability_P*_V*', 'PS_prod_P*_S*',
       'PS_contrarian_P*_S*', 'VS_prod_V*_S*', 'VS_panic_V*_S*', 'target']

In [4]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,vol_adj_market_M*_V*_squared,duration_proxy_P*_I*_squared,bull_bear_indicator,abs_M*,abs_E*,abs_S*,M*_squared,V*_squared,S*_squared,target
6969,0.041781,0.018889,0.080586,0.079422,0.205607,0.360656,0.048290,0.780305,0.733879,0.677652,...,0.982371,0.281372,0.465540,0.018793,0.199954,0.130687,0.000366,0.006037,0.017082,0.001145
6970,0.041500,0.018512,0.076923,0.075812,0.196262,0.344262,0.007713,0.780123,0.733868,0.677908,...,0.982371,0.453673,0.474814,0.009510,0.201488,0.165072,0.000097,0.005457,0.027252,0.004738
6971,0.041219,0.018134,0.073260,0.072202,0.186916,0.327869,0.007378,0.779941,0.733856,0.678165,...,0.982370,0.530169,0.463007,0.045114,0.202172,0.213920,0.002065,0.005803,0.045766,0.006016
6972,0.040938,0.017756,0.069597,0.068592,0.177570,0.311475,0.007042,0.776877,0.893087,0.678422,...,0.982370,0.416599,0.460427,0.125602,0.233134,0.249235,0.015852,0.004745,0.062123,0.001414
6973,0.040657,0.017378,0.065934,0.064982,0.168224,0.295082,0.006707,0.776701,0.892525,0.678680,...,0.982370,0.374808,0.470833,0.085487,0.227262,0.221445,0.007362,0.004698,0.049042,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.282092,0.210049,0.205128,0.202166,0.149533,0.262295,0.923541,0.627247,0.437709,0.479189,...,0.982371,0.124922,0.533316,0.044606,0.196676,0.215430,0.002019,0.002962,0.046415,0.002457
8986,0.281430,0.209671,0.201465,0.198556,0.140187,0.245902,0.923877,0.627242,0.437767,0.479013,...,0.982371,0.127873,0.556826,0.104229,0.181575,0.379219,0.010928,0.002597,0.143813,0.002312
8987,0.280770,0.209294,0.197802,0.194946,0.130841,0.229508,0.924212,0.627200,0.437777,0.478845,...,0.982371,0.094353,0.554195,0.086305,0.180759,0.321863,0.007503,0.001876,0.103601,0.002891
8988,0.280112,0.208916,0.194139,0.191336,0.121495,0.213115,0.924547,0.627159,0.437787,0.478676,...,0.982371,0.144261,0.537468,0.046895,0.187123,0.134972,0.002230,0.002095,0.018220,0.008310


In [5]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {'boosting_type': 'gbdt', 
               'objective': 'regression_l1', 
               'metric': 'mape', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 57692226.5887
Training Model 2...
Cumulative Training MAPE: 2543171.0855
Training Model 3...
Cumulative Training MAPE: 31141249.2838
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 31141249.2838
MAE: 0.0015
RMSE: 0.0042


In [6]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)


def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))